In [1]:
from __future__ import annotations
import os
import sys
from pathlib import Path

project_root = Path.cwd().parent
sys.path.insert(0, str(project_root))

import urllib.request
from typing import List, Tuple
import random
import torch
from torch.utils.data import Dataset
import re
from utils.config import KilatConfig
from data.collator import KilatDataCollator
# from data.dataset import KilatDataset
from training.trainer import KilatTrainer, TrainingArguments
from arc.model import KilatTransformer

In [2]:
def analyze_text_file(file_path):
    with open(file_path, 'r', encoding='utf-8') as file:
        text = file.read()
        
        # Basic statistics
        words = text.split()
        lines = text.splitlines()
        characters = len(text)
        characters_no_spaces = len(text.replace(' ', '').replace('\n', ''))
        
        # Count unique words (case-insensitive)
        unique_words = set(word.lower().strip('.,!?;:"()[]{}') for word in words)
        
        print(f"File: {file_path}")
        print(f"Total words: {len(words):,}")
        print(f"Total lines: {len(lines):,}")
        print(f"Total characters (with spaces): {characters:,}")
        print(f"Total characters (without spaces): {characters_no_spaces:,}")
        print(f"Unique words: {len(unique_words):,}")
        print(f"Average words per line: {len(words)/len(lines) if lines else 0:.2f}")
        
        # Show first few lines as preview
        print(f"\nFirst 100 characters preview:")
        print(text[:100] + "...")
        
        return {
            'total_words': len(words),
            'total_lines': len(lines),
            'total_characters': characters,
            'unique_words': len(unique_words)
        }

In [3]:
file_path = "./alkitab_text.txt"
stats = analyze_text_file(file_path)

File: ./alkitab_text.txt
Total words: 282,901
Total lines: 2,634
Total characters (with spaces): 1,683,221
Total characters (without spaces): 1,398,533
Unique words: 11,526
Average words per line: 107.40

First 100 characters preview:
Matius Kata-Kata Partama


Kata-Kata Partama Matius pung Kabar Bae soal Yesus ni, akang carita soal ...


In [4]:

def clean_alkitab_text(text):
    text = re.sub(r'\d+:\d+-\d+', '', text)
    text = re.sub(r'\([^)]*\d+:\d+[^)]*\)', '', text)
    text = re.sub(r'\b\d+\b', '', text)
    text = re.sub(r'#\w+\s+\d+:\d+[^\n]*', '', text)
    text = re.sub(r'#\d+:\d+[^\n]*', '', text)
    text = re.sub(r'#\d+:\d+\s*[^\n]*', '', text)
    text = re.sub(r'\s+', ' ', text)
    text = re.sub(r'\n\s*\n', '\n\n', text)
    text = text.strip()
    
    return text

def clean_alkitab_simple(text):
    text = re.sub(r'\b\d+\b', '', text)
    text = re.sub(r'\([^)]*\)', '', text)
    text = re.sub(r'#[^\n]*', '', text)
    text = re.sub(r'\s+', ' ', text)
    text = text.strip()
    
    return text

def get_clean_text_only(text):
    text = re.sub(r'Matius\s+\d+\n', '', text)
    text = re.sub(r'[A-Z][^.!?]*\d+:\d+[^\n]*\n', '', text)
    text = re.sub(r'\d+', '', text)
    text = re.sub(r'\([^)]*\)', '', text)
    text = re.sub(r'#[^\n]*', '', text)
    text = re.sub(r'\s+', ' ', text)
    text = text.strip()
    return text

with open(file_path, 'r', encoding='utf-8') as file:
    text = file.read()

cleaned_text = clean_alkitab_text(text)

words = cleaned_text.split()
print(f"Original word count: {len(text.split()):,}")
print(f"Cleaned word count: {len(words):,}")
print(f"Removed {(len(text.split()) - len(words)):,} words")
print("\n" + "="*50)
print("CLEANED TEXT PREVIEW:")
print("="*50)
print(cleaned_text[:500])

Original word count: 282,901
Cleaned word count: 281,097
Removed 1,804 words

CLEANED TEXT PREVIEW:
Matius Kata-Kata Partama Kata-Kata Partama Matius pung Kabar Bae soal Yesus ni, akang carita soal Yesus Kristus pung hidop, mulai dari Antua lahir, dapa salib, sampe hidop kombali. Matius carita Yesus kasi balajar orang, kasi bae orang yang saki, user setang dari orang yang takanal, kasi hidop orang mati, kasi ampong orang pung dosa, deng biking tanda-tanda herang. Abis samua tu, Yesus dapa tangkap, dapa siksa, deng dapa salib, mar hidop kombali pas akang pung hari katiga. Tarus, Yesus pasáng An


In [5]:
class CharTokenizer:
    def __init__(self, text: str) -> None:
        chars = sorted(list(set(text)))
        self.stoi: dict[str, int] = {ch: i + 3 for i, ch in enumerate(chars)}  # 0,1,2 reserved
        self.itos: dict[int, str] = {i: ch for ch, i in self.stoi.items()}

        self.stoi["<PAD>"] = PAD_TOKEN
        self.stoi["<BOS>"] = BOS_TOKEN
        self.stoi["<EOS>"] = EOS_TOKEN
        self.itos[PAD_TOKEN] = "<PAD>"
        self.itos[BOS_TOKEN] = "<BOS>"
        self.itos[EOS_TOKEN] = "<EOS>"

        self.vocab_size = len(self.stoi)
        self.pad_token_id = PAD_TOKEN
        self.bos_token_id = BOS_TOKEN
        self.eos_token_id = EOS_TOKEN

    def encode(self, text: str) -> list[int]:
        return [self.stoi.get(c, PAD_TOKEN) for c in text]

    def decode(self, ids: list[int]) -> str:
        return "".join(self.itos.get(i, "?") for i in ids)

In [6]:
BLOCK_SIZE: int = 256       # context length for training chunks
BATCH_SIZE: int = 32        # samples per batch (no grad accum — keep it simple)
PAD_TOKEN: int = 0
BOS_TOKEN: int = 1
EOS_TOKEN: int = 2
STRIDE: int = 32            # stride for sliding window (12.5% of BLOCK_SIZE)

class Dataset(Dataset):    
    def __init__(self, data: torch.Tensor, block_size: int = BLOCK_SIZE):
        self.data = data
        self.block_size = block_size
        self.max_start = len(data) - block_size
        
    def __len__(self) -> int:
        return max(1, self.max_start)
    
    def __getitem__(self, idx: int) -> dict:
        if self.max_start > 0:
            start = torch.randint(0, self.max_start, (1,)).item()
        else:
            start = 0
            
        chunk = self.data[start:start + self.block_size]
        return {"input_ids": chunk.tolist()}

In [7]:

def prepare_datasets(
    text: str, tokenizer: CharTokenizer, train_frac: float = 0.9
) -> Tuple[Dataset, Dataset]:
    data = torch.tensor(tokenizer.encode(text), dtype=torch.long)
    
    n = int(train_frac * len(data))
    train_data = data[:n]
    eval_data = data[n:]
    
    print(f"Total tokens: {len(data):,}")
    print(f"Train tokens: {len(train_data):,} ({(len(train_data)/len(data))*100:.1f}%)")
    print(f"Eval tokens:  {len(eval_data):,} ({(len(eval_data)/len(data))*100:.1f}%)")
    print(f"Possible train chunks: {max(0, len(train_data) - BLOCK_SIZE):,}")
    print(f"Possible eval chunks:  {max(0, len(eval_data) - BLOCK_SIZE):,}")
    
    train_dataset = Dataset(train_data, block_size=BLOCK_SIZE)
    eval_dataset = Dataset(eval_data, block_size=BLOCK_SIZE)
    return train_dataset, eval_dataset

In [8]:
def build_model(vocab_size: int) -> KilatTransformer:
    config = KilatConfig(
        vocab_size=vocab_size,
        n_embd=384,
        n_layer=6,
        n_head=6,
        recall_ratio=0.5,
        latent_dim=32,
        ffn_mode="dense",
        ff_mult=8 / 3,
        embd_drop=0.1,
        attn_drop=0.1,
        ffn_dropout=0.1,
        resid_drop=0.1,
        pad_token_id=PAD_TOKEN,
        bos_token_id=BOS_TOKEN,
        eos_token_id=EOS_TOKEN,
        tie_word_embeddings=True,
    )

    model = KilatTransformer(config)
    total_params = sum(p.numel() for p in model.parameters())
    trainable_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
    print(f"Vocabulary size:     {vocab_size:,}")
    print(f"Total parameters:    {total_params:,}")
    print(f"Trainable parameters: {trainable_params:,}")
    return model

In [9]:

@torch.inference_mode()
def generate(
    model: KilatTransformer,
    tokenizer: CharTokenizer,
    prompt: str,
    max_new_tokens: int = 500,
    temperature: float = 0.7,
    top_k: int = 15,                 
    top_p: float = 0.85,            
    repetition_penalty: float = 1.15, 
    device: str = "cpu",
) -> str:
    model.eval()
    input_ids = torch.tensor([tokenizer.encode(prompt)], dtype=torch.long).to(device)

    for _ in range(max_new_tokens):
        if input_ids.size(1) > BLOCK_SIZE:
            input_ids = input_ids[:, -BLOCK_SIZE:]

        outputs = model(input_ids, return_dict=True)
        logits = outputs.logits[:, -1, :].clone()  

        if temperature == 0.0:
            next_token = torch.argmax(logits, dim=-1, keepdim=True)
        else:
            if repetition_penalty != 1.0:
                for score_idx in range(logits.size(0)):
                    # Ambil token unik yang sudah digenerasikan sejauh ini
                    generated_tokens = set(input_ids[score_idx].tolist())
                    for token_id in generated_tokens:
                        # Jika logit > 0, turunkan nilainya; jika < 0, buat semakin negatif
                        if logits[score_idx, token_id] > 0:
                            logits[score_idx, token_id] /= repetition_penalty
                        else:
                            logits[score_idx, token_id] *= repetition_penalty

            logits = logits / temperature
            
            if top_k > 0:
                top_k_values, _ = torch.topk(logits, min(top_k, logits.size(-1)))
                kth_values = top_k_values[..., -1, None]
                logits[logits < kth_values] = float('-inf')
            
            if top_p > 0.0 and top_p < 1.0:
                sorted_logits, sorted_indices = torch.sort(logits, descending=True, dim=-1)
                cumulative_probs = torch.cumsum(torch.softmax(sorted_logits, dim=-1), dim=-1)
                sorted_indices_to_remove = cumulative_probs > top_p
                sorted_indices_to_remove[..., 1:] = sorted_indices_to_remove[..., :-1].clone()
                sorted_indices_to_remove[..., 0] = 0
                indices_to_remove = sorted_indices_to_remove.scatter(
                    dim=-1, index=sorted_indices, src=sorted_indices_to_remove
                )
                logits[indices_to_remove] = float('-inf')
            probs = torch.softmax(logits, dim=-1)
            next_token = torch.multinomial(probs, num_samples=1)

        input_ids = torch.cat([input_ids, next_token], dim=-1)

        if next_token.item() == EOS_TOKEN:
            break

    return tokenizer.decode(input_ids[0].tolist())

In [ ]:
def main() -> None:
    device = "cuda" if torch.cuda.is_available() else "cpu"
    print(f"Using device: {device}\n")
    tokenizer = CharTokenizer(cleaned_text)
    
    print(f"Vocabulary size: {tokenizer.vocab_size} unique characters (+3 special tokens)")
    print(f"Special tokens: <PAD>={PAD_TOKEN}, <BOS>={BOS_TOKEN}, <EOS>={EOS_TOKEN}")
    print()

    train_dataset, eval_dataset = prepare_datasets(cleaned_text, tokenizer, train_frac=0.9)
    
    print(f"\nTraining dataset size (epochs worth): {len(train_dataset):,} unique chunks")
    print(f"Evaluation dataset size: {len(eval_dataset):,} unique chunks")
    print(f"Each epoch will sample ~{min(len(train_dataset), 5000):,} random chunks")
    print()

    data_collator = KilatDataCollator(
        pad_token_id=PAD_TOKEN,
        max_length=BLOCK_SIZE,
        ignore_index=-100,
    )

    model = build_model(tokenizer.vocab_size)
    model.to(device)
    total_steps = 5000
    tokens_per_step = BATCH_SIZE * BLOCK_SIZE
    total_tokens_seen = total_steps * tokens_per_step
    total_unique_tokens = len(train_dataset.data) - BLOCK_SIZE
    
    print(f"\nTraining statistics:")
    print(f"  Total training steps: {total_steps:,}")
    print(f"  Tokens per step: {tokens_per_step:,}")
    print(f"  Total tokens processed: {total_tokens_seen:,}")
    print(f"  Unique tokens available: {total_unique_tokens:,}")
    print(f"  Coverage: {min(100.0, (total_tokens_seen / total_unique_tokens) * 100):.1f}% of possible chunks")
    print()

    training_args = TrainingArguments(
        output_dir="./tmp_kilat",
        save_checkpoints=False,
        training_mode="steps",
        max_steps=total_steps,
        learning_rate=3e-4,
        per_device_train_batch_size=BATCH_SIZE,
        per_device_eval_batch_size=BATCH_SIZE,
        gradient_accumulation_steps=1,
        weight_decay=0.01,
        max_grad_norm=1.0,
        warmup_steps=200,
        logging_steps=100,
        eval_steps=500,
        save_steps=1_000_000_000,
        save_total_limit=0,
        early_stopping_patience=0,
        precision="bf16" if device == "cuda" else "fp32",
        report_to="none",
        run_name="kilat-shakespeare",
        seed=1337,
    )

    trainer = KilatTrainer(
        model=model,
        args=training_args,
        train_dataset=train_dataset,
        eval_dataset=eval_dataset,
        data_collator=data_collator,
    )
    trainer.train()

    print("\n" + "=" * 60)
    print("Generating samples with optimized sampling strategy...")
    print("=" * 60 + "\n")

    prompts = [
        "Matius",
        "Barang",
        "Beta Su",
        "    ",
        "Kanapa",
        "Ose",
    ]

    for prompt in prompts:
        print(f"--- Prompt: \"{prompt.strip()}\" ---")
        generated = generate(
            model, tokenizer, prompt,
            max_new_tokens=500,
            temperature=0.7,
            top_k=15,            
            top_p=0.85,           
            repetition_penalty=1.15, 
            device=device,
        )
        print(generated)
        print()
        print("-" * 60)
        print()

    print("Done.")


In [11]:
main()

Using device: cuda

Vocabulary size: 79 unique characters (+3 special tokens)
Special tokens: <PAD>=0, <BOS>=1, <EOS>=2

Total tokens: 1,653,745
Train tokens: 1,488,370 (90.0%)
Eval tokens:  165,375 (10.0%)
Possible train chunks: 1,488,114
Possible eval chunks:  165,119

Training dataset size (epochs worth): 1,488,114 unique chunks
Evaluation dataset size: 165,119 unique chunks
Each epoch will sample ~5,000 random chunks

Vocabulary size:     79
Total parameters:    11,357,970
Trainable parameters: 11,357,970

Training statistics:
  Total training steps: 5,000
  Tokens per step: 8,192
  Total tokens processed: 40,960,000
  Unique tokens available: 1,488,114
  Coverage: 100.0% of possible chunks


KilatTransformer Training
Output dir:          ./tmp_kilat
Save checkpoints:    False
Training mode:       steps
Total target steps:  5,000
Batch size (per GPU):32
Gradient accum:      1
Effective batch:     32
Learning rate:       3.00e-04
Warmup steps:        200
Weight decay:        0.01
Ma

Training (steps):   2%|▏         | 100/5000 [00:24<15:31,  5.26step/s, loss=1.7447, ppl=5.7, lr=1.50e-04] 

[13:58:17] Step:    100/ 5,000 | Loss:   1.7447 | PPL:      5.7 | LR: 1.50e-04 | Grad:   1.37 | Speed:    4.1 st/s | Remaining:   19.8 min


Training (steps):   4%|▍         | 201/5000 [00:43<14:45,  5.42step/s, loss=1.4276, ppl=4.2, lr=3.00e-04]

[13:58:36] Step:    200/ 5,000 | Loss:   1.3426 | PPL:      3.8 | LR: 3.00e-04 | Grad:   1.22 | Speed:    4.6 st/s | Remaining:   17.4 min


...

[14:26:16] Step:  2,900/ 5,000 | Loss:   0.6070 | PPL:      1.8 | LR: 1.21e-04 | Grad:   0.78 | Speed:    1.7 st/s | Remaining:   20.6 min


Training (steps):  60%|██████    | 3000/5000 [28:34<04:04,  8.18step/s, loss=0.6709, ppl=2.0, lr=1.11e-04]

[14:26:28] Step:  3,000/ 5,000 | Loss:   0.6709 | PPL:      2.0 | LR: 1.11e-04 | Grad:   0.87 | Speed:    1.7 st/s | Remaining:   19.1 min


Training (steps):  60%|██████    | 3000/5000 [31:43<21:09,  1.58step/s, loss=0.6709, ppl=2.0, lr=1.11e-04]



[Eval sample prompt] [token ids] 49 67 68 60 8 3 50 57 49 65 3 52 63 62 55 3 66 49 60 49 61 49 67 10 3 35 49 60 63 3 59 49 61 63 62 55 3 50 57 59 57 62 55 3 50 49 55 57 67 68 8 3 57 67 68 3 66 49 61 49 3 66 49 3 52 53 62 55 3 59 49 61 63 62 55 3 66 68 3 51 57 55 57 3 52 63 62 55 3 59 49 60 68 49 65 3 52 49 65 57 3 49 64 57 10 3 35 49 61 63 62 55 3 58 68 49 3 61 68 66 67 57 3 59 49 66 57 3 68 62 58 68 3 59 49 61 63 62 55 3 64 68 62 55 3 56 49 67 57 3 50 49 53 3 64 49 65 3 63 65 49 62 55 3 60 49 53 62 55 3 60 49 57 8 3 61 49 65 3 59 49 61 63 62 55 3 61 68 66 67 57 3 49 67 57 9 49 67 57 3 50 57 49 65 3 59 49 61 63 62 55 3 66 53 62 55 3 49 62 70 63 3 57 59 63 3 52 63 62 55 3 64 68 62 55 3 51 49 65 49 3 56 57 52 63 64 3 70 49 62 55 3 58 49 56 49

----------------------------------------
Evaluation Summary
----------------------------------------
Validation Loss: 0.9815
Validation PPL:  2.7
Best Val Loss:   0.9733
Step:            3000
----------------------------------------


[EarlyStoppi